# FracAtlas release: ONNX export and Hugging Face upload (CPU)

Companion notebook for [MHKhan50/fracatlas-fracture-benchmark](https://github.com/MHKhan50/fracatlas-fracture-benchmark).
It takes the output of the training notebook, exports the ConvNeXt-Tiny classifier (the seed chosen on validation
AUROC) and the YOLO11s detector to ONNX, checks the export against PyTorch, and fits the temperature and threshold
of that seed on validation data.

**Inputs (Add Input):** the output of the notebook *FracAtlas fracture detection: ConvNeXt, DINOv2 and YOLO*, and the
FracAtlas dataset used there. Accelerator: **None (CPU)**. Internet **on**.

The last cell publishes to Hugging Face only when `PUBLISH = True` and a Kaggle secret `HF_TOKEN` (write token) exists.

In [ ]:
import os, subprocess, sys, glob
os.chdir("/kaggle/working")
if not os.path.exists("fracatlas-fracture-benchmark"):
    subprocess.run(["git", "clone", "-q", "https://github.com/MHKhan50/fracatlas-fracture-benchmark.git"], check=True)
os.chdir("fracatlas-fracture-benchmark")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[train]", "onnx", "onnxruntime", "huggingface_hub"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 1. Find the training run and the images

In [ ]:
found = sorted(glob.glob("/kaggle/input/**/analysis.json", recursive=True))
assert found, "Attach the output of the training notebook (Add Input -> Your Work)."
REPORTS = os.path.dirname(found[0])
print("reports:", REPORTS, sorted(os.listdir(REPORTS)))
assert os.path.exists(REPORTS + "/weights/convnext_tiny_best.pt"), "weights missing in the attached output"
csv = sorted(glob.glob("/kaggle/input/**/dataset.csv", recursive=True))
assert csv, "Attach the FracAtlas dataset (it contains dataset.csv)."
IMAGES = os.path.dirname(csv[0])
print("images:", IMAGES)

## 2. Export to ONNX, check against PyTorch, fit temperature and threshold on validation

In [ ]:
subprocess.run([sys.executable, "-m", "fracatlas_bench.release", "--reports", REPORTS, "--out", "/kaggle/working/release"], check=True)
for f in sorted(os.listdir("/kaggle/working/release")):
    print(f, round(os.path.getsize("/kaggle/working/release/" + f) / 1e6, 1), "MB")

## 3. Near-duplicate check between splits (perceptual hashes; a few minutes)

In [ ]:
subprocess.run([sys.executable, "-m", "fracatlas_bench.neardup", "--images", IMAGES, "--reports", REPORTS,
                "--out", "/kaggle/working/neardup"], check=True)
from IPython.display import Image as Show
Show("/kaggle/working/neardup/closest_pairs.png")

## 4. Publish (only when you decide to)

In [ ]:
PUBLISH = False   # set to True to create/update the public Hugging Face model and Space
if PUBLISH:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    subprocess.run([sys.executable, "scripts/publish_hf.py", "--release", "/kaggle/working/release",
                    "--split", REPORTS + "/split.csv", "--images", IMAGES], check=True)
else:
    print("Not published. Review /kaggle/working/release/config.json, then set PUBLISH = True and run this cell.")